# 02 - Doubly robust AIPW

This notebook compares outcome regression, IPW, and augmented inverse probability weighting.

AIPW is called doubly robust because, under standard conditions, it can remain consistent if either the treatment model or the outcome model is correctly specified.


## Causal question
State the causal question this notebook answers before reading numeric output.

## Causal setup
- Treatment: define the treatment variable and intervention of interest.
- Outcome: define the outcome variable being affected by treatment.
- Covariates: list observed confounders included in the design/diagnostics.
- Unit of analysis: specify the observational unit used in this notebook.

## Estimand
Specify the target estimand (ATE, ATT, CATE, etc.) and how it maps to model coefficients.

## Identification assumptions
Enumerate the identification assumptions required for a causal interpretation (for example ignorability, overlap, no interference).

## Uncertainty and limitations
Report interval estimates, sensitivity checks, and at least one limitation of the design.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.estimators import (
    aipw_ate,
    difference_in_means,
    g_computation_ate,
    ipw_ate,
)
from causal_inference_lab.uncertainty import bootstrap_ate

dataset = make_confounded_binary_treatment(n=5_000, seed=123)
data = dataset.data
covariates = ["x1", "x2", "x3"]


## Compare estimators

We compare each estimator against the known true ATE.


In [ ]:
estimates = [
    difference_in_means(data),
    g_computation_ate(data, covariates),
    ipw_ate(data, covariates),
    aipw_ate(data, covariates),
]

results = pd.DataFrame(
    {
        "estimator": [estimate.estimator for estimate in estimates],
        "estimate": [estimate.estimate for estimate in estimates],
        "true_ate": dataset.true_ate,
        "absolute_error": [abs(estimate.estimate - dataset.true_ate) for estimate in estimates],
    }
)

results


## Bootstrap uncertainty interval

This uses the reusable `bootstrap_ate` function and reports a percentile confidence interval for AIPW under repeated resampling. The interval is for uncertainty quantification only and does not replace identification assumptions.


In [ ]:
bootstrap_result = bootstrap_ate(
    data=data,
    estimator=lambda frame: aipw_ate(frame, covariates),
    n_bootstrap_samples=500,
    seed=42,
    confidence_level=0.95,
)

print(f"AIPW estimate: {bootstrap_result.estimate:.3f}")
print(
    f"{int(bootstrap_result.confidence_level*100)}% bootstrap interval: "
    f"[{bootstrap_result.lower:.3f}, {bootstrap_result.upper:.3f}]"
)
print(f"Standard error: {bootstrap_result.std_error:.3f}")
print(f"True ATE: {dataset.true_ate:.3f}")


**Interpretation.** AIPW is a strong default for observational causal analysis, but it still depends on the causal assumptions. Double robustness is not protection against unmeasured confounding.
